# Confidence calibration

How well the model's stated `confidence` matches how often it is right, for one `evaluate` sweep.

After https://arxiv.org/pdf/2512.16030

In [1]:
%matplotlib ipympl

In [8]:
from analysis.build_data import load_trace_df
from analysis.calibration import (
    category_accuracy_table,
    category_table,
    confidence_metrics,
    expected_calibration_error,
    horizon_table,
    kalshibench_metrics,
)


In [10]:
# one scoreable forecast per row, from every child run of the sweep (see analysis/build_data.py)
forecasts = load_trace_df("outcome-confidence", run_name="sweep-20261106-1744")

## Dataset statistics

Table 2 of the paper: the markets in the sample by category, with the share resolving to the priced outcome (see `category_table` in `analysis/calibration.py`).

In [54]:
category_table(forecasts).round(1)

,N,%,Yes%
category,,,
politics,984,50.8,23.5
finance,452,23.3,44.2
economics,448,23.1,41.3
elections,42,2.2,28.6
companies,11,0.6,45.5


## Expected calibration error

Forecasts binned by stated confidence (`confidence_bin`, from `load_trace_df`; see `expected_calibration_error` in `analysis/calibration.py`).

In [ ]:
ece = expected_calibration_error(forecasts)

In [ ]:
ece_by_model = forecasts.groupby("model").apply(
    expected_calibration_error, include_groups=False
)

## KalshiBench table

Table 4 of the paper, per model (see `kalshibench_metrics` in `analysis/calibration.py` for the columns).

In [3]:
model_table = forecasts.groupby("model").apply(
    kalshibench_metrics, include_groups=False
)

In [4]:
model_table.round(3)

,N,Acc,F1,F1yes,Brier,BSS,ECE,MCE
model,,,,,,,,
claude-fable-5-1,441.0,71.429,0.632,0.457,0.183,0.181,0.032,0.101
claude-opus-4-5,997.0,69.208,0.628,0.473,0.210,0.038,0.046,0.350
claude-opus-5-5,991.0,71.140,0.625,0.446,0.183,0.177,0.029,0.070


## Confidence analysis

Table 5 of the paper, per model: average confidence, confidence when wrong, and the overconfidence rate above 70/80/90% (see `confidence_metrics` in `analysis/calibration.py`).

In [9]:
confidence_table = forecasts.groupby("model").apply(
    confidence_metrics, include_groups=False
)

In [10]:
confidence_table.round(1)

,Avg Conf,Conf_wrong,OCR@70,OCR@80,OCR@90
model,,,,,
claude-fable-5-1,71.0,62.9,12.7,12.7,9.6
claude-opus-4-5,70.9,67.3,23.4,21.5,22.6
claude-opus-5-5,70.1,62.0,12.9,9.9,3.7


In [5]:
# the reliability table behind the number: per bin, its size, conf(B_b) and acc(B_b)
reliability = forecasts.groupby("confidence_bin", observed=True).agg(
    n=("correct", "size"), conf=("confidence", "mean"), acc=("correct", "mean")
)

## Forecast horizon

Accuracy and confidence by days from the model's training cutoff to the market's close (see `horizon_table` in `analysis/calibration.py`).

In [3]:
horizon_table(forecasts).round(2)

N    Acc  Avg Conf   ECE
model                                                      
claude-fable-5-1 (-0.001, 30.0]   50   78.0     78.00  0.11
                 (30.0, 60.0]     61  85.25     78.15  0.12
                 (60.0, 90.0]    137  72.99     72.15  0.05
                 (90.0, 180.0]   187  65.24     65.97  0.05
                 (180.0, 365.0]    4   25.0     73.50  0.50
                 (365.0, inf]      2   50.0     74.00  0.36
claude-opus-4-5  (-0.001, 30.0]   16  56.25     75.50  0.22
                 (30.0, 60.0]     21  85.71     75.86  0.18
                 (60.0, 90.0]     42   61.9     75.83  0.22
                 (90.0, 180.0]   150   70.0     76.11  0.10
                 (180.0, 365.0]  467  72.16     70.92  0.04
                 (365.0, inf]    299  64.55     66.95  0.07
claude-opus-5-5  (-0.001, 30.0]  104  79.81     78.34  0.07
                 (30.0, 60.0]    128  79.69     79.63  0.02
                 (60.0, 90.0]    306  71.57     71.31  0.02
                 (90.0, 180.0]   445  66.74     64.47  0.04
                 (180.0, 365.0]    4   25.0     74.50  0.51
                 (365.0, inf]      4   75.0     83.00  0.26

## Market category

Accuracy and confidence by model and market category (see `category_accuracy_table` in `analysis/calibration.py`).

In [11]:
category_accuracy_table(forecasts).round(2)

NameError: name 'accuracy_and_confidence' is not defined